In [1]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
from tqdm import tqdm

import torch
from lhotse import CutSet
from lhotse.dataset import SimpleCutSampler
from lhotse.dataset.input_strategies import OnTheFlyFeatures

from transformers import AutoTokenizer, AutoFeatureExtractor

from data.audio.lhotse import LibriSpeechLhotse, PeoplesSpeechLhotse
from melt.processing_melt import MELTProcessor

/mnt/home/giuseppe/mydata/melt-proj/training/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Set the paths to your Shar directories for LibriSpeech and People's Speech datasets.

In [2]:
# Update these paths to match your data location
LIBRISPEECH_SHAR_DIR = Path("/mnt/home/giuseppe/myscratch/melt-data/shar/librispeech")
PEOPLES_SPEECH_SHAR_DIR = Path("/mnt/home/giuseppe/myscratch/melt-data/shar/peoples_speech")

# Model for the processor
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Or any tokenizer you want
AUDIO_ENCODER_NAME = "facebook/w2v-bert-2.0"  # Or any feature extractor

# Number of samples to iterate over
NUM_SAMPLES = 500

# Random seed for reproducibility
SEED = 42

## 1. Load LibriSpeech and People's Speech CutSets

In [3]:
# Initialize dataset loaders
librispeech = LibriSpeechLhotse(LIBRISPEECH_SHAR_DIR)
peoples_speech = PeoplesSpeechLhotse(PEOPLES_SPEECH_SHAR_DIR)

print(f"LibriSpeech configs: {librispeech.get_available_configs()}")
print(f"People's Speech configs: {peoples_speech.get_available_configs()}")

LibriSpeech configs: ['clean', 'other']
People's Speech configs: ['clean', 'dirty', 'clean_sa', 'dirty_sa', 'microset']


In [4]:
# Load training cuts from both datasets
# Using validation sets for faster iteration (smaller)
librispeech_cuts = librispeech.load_cuts(
    split="validation",
    config="clean",
    shuffle_shards=True,
    seed=SEED,
)

peoples_speech_cuts = peoples_speech.load_cuts(
    split="validation",
    config="clean",
    shuffle_shards=True,
    seed=SEED,
)

print(f"Loaded LibriSpeech CutSet: {librispeech_cuts}")
print(f"Loaded People's Speech CutSet: {peoples_speech_cuts}")

Loaded LibriSpeech CutSet: CutSet(len=2703) [underlying data type: <class 'lhotse.shar.readers.lazy.LazySharIterator'>]
Loaded People's Speech CutSet: CutSet(len=18622) [underlying data type: <class 'lhotse.shar.readers.lazy.LazySharIterator'>]


## 2. Concatenate and Shuffle CutSets

We use `CutSet.mux()` for interleaving from multiple sources, or simply concatenate with `+` and then shuffle.

In [6]:
# Option 1: Simple concatenation and shuffle
# Note: For lazy CutSets, this creates a lazy chain
combined_cuts = (librispeech_cuts + peoples_speech_cuts).shuffle()

# Option 2: Use mux for weighted interleaving (uncomment to use)
# This interleaves samples from both datasets according to weights
# combined_cuts = CutSet.mux(
#     librispeech_cuts,
#     peoples_speech_cuts,
#     weights=[0.5, 0.5],  # Equal sampling probability
#     seed=SEED,
# )

print(f"Combined CutSet: {combined_cuts}")

Combined CutSet: CutSet(len=21325) [underlying data type: <class 'lhotse.lazy.LazyShuffler'>]


## 3. Initialize MELTProcessor

Create the processor with the tokenizer and feature extractor.

In [7]:
# Load tokenizer and feature extractor
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
feature_extractor = AutoFeatureExtractor.from_pretrained(AUDIO_ENCODER_NAME)

# Ensure tokenizer has a pad token
if tokenizer.pad_token is None:
    print("Setting pad token to eos token")
    tokenizer.pad_token = tokenizer.eos_token

# Create MELTProcessor
processor = MELTProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print(f"Processor created with:")
print(f"  - Tokenizer vocab size: {len(tokenizer)}")
print(f"  - Audio token: {processor.audio_token}")
print(f"  - Audio BOS token: {processor.audio_bos_token}")
print(f"  - Audio EOS token: {processor.audio_eos_token}")

Processor created with:
  - Tokenizer vocab size: 151672
  - Audio token: <|AUDIO|>
  - Audio BOS token: <|audio_bos|>
  - Audio EOS token: <|audio_eos|>


## 4. Create Custom Dataset with OnTheFlyFeatures

We'll create a simple dataset that uses `OnTheFlyFeatures` for audio processing and MELTProcessor for the final batch preparation.

In [8]:
from torch.utils.data import Dataset, DataLoader
from lhotse.features.base import FeatureExtractor
from lhotse.dataset.collation import collate_audio


class MELTASRDataset(Dataset):
    """A simple ASR dataset that uses MELTProcessor for batch preparation.
    
    This dataset:
    1. Receives a CutSet batch from the sampler
    2. Loads audio on-the-fly
    3. Uses MELTProcessor to prepare inputs for the model
    """
    
    def __init__(
        self,
        processor: MELTProcessor,
        prompt_template: str = "Transcribe the following audio: {audio_token}",
    ):
        self.processor = processor
        self.prompt_template = prompt_template
    
    def __getitem__(self, cuts: CutSet) -> dict:
        """Process a batch of cuts.
        
        Args:
            cuts: A CutSet containing the batch of cuts to process.
            
        Returns:
            A dictionary with processed inputs ready for the model.
        """
        # Load audio for all cuts
        audios = []
        texts = []
        transcripts = []
        
        for cut in cuts:
            # Load audio
            audio = cut.load_audio()  # Shape: (channels, samples)
            if audio.ndim == 2:
                audio = audio.squeeze(0)  # Remove channel dim if mono
            audios.append(audio)
            
            # Get transcript
            transcript = cut.supervisions[0].text if cut.supervisions else ""
            transcripts.append(transcript)
            
            # Prepare text with audio token placeholder
            text = self.prompt_template.format(audio_token=self.processor.audio_token)
            texts.append(text)
        
        # Use MELTProcessor to prepare the batch
        # Each sample has one audio, so audio is a list of lists (one audio per sample)
        audio_inputs = [[a] for a in audios]  # List of lists for batched processing
        
        batch = self.processor(
            text=texts,
            audio=audio_inputs,
            return_tensors="pt",
            padding=True,
        )
        
        # Add transcripts for reference
        batch["transcripts"] = transcripts
        
        return batch

In [ ]:
# Alternative: Using Lhotse's OnTheFlyFeatures directly
# This shows how to use OnTheFlyFeatures with a custom FeatureExtractor wrapper

from lhotse.features import Fbank, FbankConfig

# Create a Lhotse feature extractor (e.g., Filter Bank features)
# You could also create a custom extractor that wraps MELTProcessor's feature extractor
lhotse_extractor = Fbank(FbankConfig(num_mel_bins=80))

# OnTheFlyFeatures computes features during iteration
on_the_fly = OnTheFlyFeatures(
    extractor=lhotse_extractor,
    num_workers=0,  # Set > 0 for parallel feature extraction
)

print(f"OnTheFlyFeatures extractor: {on_the_fly}")

## 5. Iterate Over Samples and Compute Features

In [ ]:
from lhotse.dataset import DynamicBucketingSampler

# Create dataset and sampler
dataset = MELTASRDataset(
    processor=processor,
    prompt_template="<|begin_of_text|>Transcribe: {audio_token}<|end_of_text|>",
)

# SimpleCutSampler samples cuts up to max_duration seconds per batch
sampler = DynamicBucketingSampler(
    combined_cuts,
    max_duration=240,
    num_buckets=10,
    shuffle=True,
    seed=SEED,
)

# Create DataLoader
# Note: batch_size=None because the sampler handles batching
dataloader = DataLoader(
    dataset,
    sampler=sampler,
    batch_size=None,
    num_workers=0,  # Set > 0 for parallel data loading
)

In [ ]:
# Iterate over the first NUM_SAMPLES samples
sample_count = 0
batch_count = 0

print(f"Iterating over {NUM_SAMPLES} samples...\n")

for batch in tqdm(dataloader, desc="Processing batches"):
    batch_size = batch["input_ids"].shape[0]
    sample_count += batch_size
    batch_count += 1
    
    # Print info for first few batches
    if batch_count <= 3:
        print(f"\nBatch {batch_count}:")
        print(f"  - Batch size: {batch_size}")
        print(f"  - input_ids shape: {batch['input_ids'].shape}")
        print(f"  - attention_mask shape: {batch['attention_mask'].shape}")
        if "input_features" in batch:
            print(f"  - input_features shape: {batch['input_features'].shape}")
        if "features_attention_mask" in batch:
            print(f"  - features_attention_mask shape: {batch['features_attention_mask'].shape}")
        if "audio_lengths" in batch:
            print(f"  - audio_lengths: {batch['audio_lengths']}")
        print(f"  - Sample transcripts: {batch['transcripts'][:2]}...")
    
    if sample_count >= NUM_SAMPLES:
        print(f"\nReached {sample_count} samples after {batch_count} batches.")
        break

print(f"\nTotal samples processed: {sample_count}")
print(f"Total batches processed: {batch_count}")

Iterating over 500 samples...



Processing batches: 1it [00:34, 34.20s/it]


Batch 1:
  - Batch size: 4
  - input_ids shape: torch.Size([4, 748])
  - attention_mask shape: torch.Size([4, 748])
  - input_features shape: torch.Size([4, 732, 160])
  - feature_attention_mask shape: torch.Size([4, 732])
  - audio_lengths: tensor([[682],
        [728],
        [676],
        [714]])
  - Sample transcripts: ["second of all you have a property that's in the floodplain that's owned by one owner that might just do repairs to windows and door that then sells the property to another property owner", "so are they not referring back to that well and that works if you've got everybody at the table sitting down when the loan is executed and you pass the documents around and it's executed contemporaneously the guarantor can leave with a copy of the note"]...


Processing batches: 2it [00:46, 21.08s/it]


Batch 2:
  - Batch size: 4
  - input_ids shape: torch.Size([4, 248])
  - attention_mask shape: torch.Size([4, 248])
  - input_features shape: torch.Size([4, 232, 160])
  - feature_attention_mask shape: torch.Size([4, 232])
  - audio_lengths: tensor([[211],
        [220],
        [228],
        [200]])
  - Sample transcripts: ["and as a result i think we'll be stronger together as a lakeshore region", 'SO HERE IT WAS SPREAD OUT CLEAR BEFORE HIM AND NOW HE KNEW WHAT TO EXPECT']...


Processing batches: 3it [00:46, 11.77s/it]


Batch 3:
  - Batch size: 4
  - input_ids shape: torch.Size([4, 169])
  - attention_mask shape: torch.Size([4, 169])
  - input_features shape: torch.Size([4, 152, 160])
  - feature_attention_mask shape: torch.Size([4, 152])
  - audio_lengths: tensor([[141],
        [ 80],
        [149],
        [ 40]])
  - Sample transcripts: ['that is what this board is going to do from this point forward', "that's probably true but we don't care"]...


Processing batches: 47it [07:48,  9.96s/it]


KeyboardInterrupt: 

## 6. Alternative: Direct Iteration with OnTheFlyFeatures

If you want to use Lhotse's `OnTheFlyFeatures` directly (e.g., for filter bank features), here's how:

In [ ]:
from lhotse.dataset import K2SpeechRecognitionDataset

# K2SpeechRecognitionDataset with OnTheFlyFeatures
k2_dataset = K2SpeechRecognitionDataset(
    return_cuts=True,
    input_strategy=on_the_fly,
)

# Create a new sampler (samplers are single-use iterators)
k2_sampler = SimpleCutSampler(
    combined_cuts,
    max_duration=30.0,
    shuffle=True,
    seed=SEED + 1,  # Different seed for variety
)

k2_dataloader = DataLoader(
    k2_dataset,
    sampler=k2_sampler,
    batch_size=None,
    num_workers=0,
)

In [ ]:
# Iterate and show feature shapes
print("Using K2SpeechRecognitionDataset with OnTheFlyFeatures:\n")

for i, batch in enumerate(k2_dataloader):
    if i >= 3:
        break
    
    print(f"Batch {i + 1}:")
    print(f"  - inputs (features) shape: {batch['inputs'].shape}")
    print(f"  - input_lens: {batch['supervisions']['num_frames'][:5]}...")
    print(f"  - Number of cuts: {len(batch['supervisions']['cut'])}")
    print()

## Summary

This notebook demonstrated:

1. **Loading datasets**: Using `LibriSpeechLhotse` and `PeoplesSpeechLhotse` to load CutSets from Shar archives

2. **Combining CutSets**: Using `+` operator and `.shuffle()` to combine and randomize data from multiple sources

3. **MELTProcessor integration**: Creating a custom dataset that uses `MELTProcessor` for preparing text+audio inputs

4. **OnTheFlyFeatures**: Using Lhotse's `OnTheFlyFeatures` strategy for computing features during iteration

5. **Efficient iteration**: Using `SimpleCutSampler` for dynamic batching based on audio duration